# 06. Formulación MILP de la capa Chi

Este notebook estudia la formulación MILP de la capa no lineal `chi`
de Keccak.

El estado de entrada corresponde a la salida de las capas:

$$
\theta
\longrightarrow
\rho
\longrightarrow
\pi.
$$

Para cada bit del estado, la transformación `chi` se define como:

$$
C[x,y,k]
=
B[x,y,k]
\oplus
\left(
\neg B[(x+1)\bmod 5,y,k]
\land
B[(x+2)\bmod 5,y,k]
\right),
$$

donde:

- $B$ es la salida de `rho` y `pi`;
- $C$ es la salida de `chi`;
- $x,y \in \{0,1,2,3,4\}$;
- $k \in \{0,1,\ldots,z-1\}$.

A diferencia de las capas lineales anteriores, `chi` combina:

- una negación;
- una conjunción lógica;
- una operación XOR.

Por esta razón, su formulación MILP puede requerir variables binarias
auxiliares y restricciones adicionales.

El objetivo del notebook será:

1. inspeccionar la formulación implementada;
2. analizar su tamaño;
3. comprobar su idempotencia;
4. resolver estados de entrada controlados;
5. comparar la salida MILP con la implementación de referencia;
6. validar varios estados pseudoaleatorios reproducibles.

In [1]:
# ============================================================
# CONFIGURACIÓN DEL ENTORNO DEL NOTEBOOK
# ============================================================

from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """
    Busca hacia arriba el directorio raíz del proyecto.

    Se considera raíz el directorio que contiene la carpeta `src`.
    """
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src").exists():
            return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto con una carpeta `src`."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


print("Raíz del proyecto:", PROJECT_ROOT)
print("Directorio src:", SRC_DIR)

Raíz del proyecto: D:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada
Directorio src: D:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada\src


In [2]:
# ============================================================
# IMPORTACIONES PRINCIPALES
# ============================================================

import inspect
import numpy as np

import keccak_milp.layers as layers

from keccak_milp.config import ExperimentConfig
from keccak_milp.model import KeccakMILPModel


print("NumPy:", np.__version__)
print("KeccakMILPModel importado correctamente.")

NumPy: 2.5.1
KeccakMILPModel importado correctamente.


In [3]:
# ============================================================
# INSPECCIÓN DE LA API RELACIONADA CON CHI
# ============================================================

import inspect

import keccak_milp.layers as layers

from keccak_milp.model import KeccakMILPModel


layer_names = [
    name
    for name in dir(layers)
    if "chi" in name.lower()
]

model_names = [
    name
    for name in dir(KeccakMILPModel)
    if "chi" in name.lower()
]


print("Funciones disponibles en keccak_milp.layers:")

for name in layer_names:
    print(" -", name)


print("\nMétodos disponibles en KeccakMILPModel:")

for name in model_names:
    print(" -", name)


required_layer_names = {
    "chi",
}

required_model_names = {
    "_add_chi_constraints",
    "_create_chi_variables",
    "add_chi_layer",
    "chi_output_values",
    "chi_output_variable",
}

assert required_layer_names.issubset(
    set(layer_names)
)

assert required_model_names.issubset(
    set(model_names)
)

print("\nLa API de Chi está disponible correctamente.")

Funciones disponibles en keccak_milp.layers:
 - chi

Métodos disponibles en KeccakMILPModel:
 - _add_chi_constraints
 - _create_chi_variables
 - add_chi_layer
 - chi_output_values
 - chi_output_variable

La API de Chi está disponible correctamente.


In [4]:
# ============================================================
# FIRMAS DE LA API DE CHI
# ============================================================

chi = layers.chi

objects_to_inspect = {
    "chi": chi,
    "KeccakMILPModel.add_chi_layer": (
        KeccakMILPModel.add_chi_layer
    ),
    "KeccakMILPModel.chi_output_variable": (
        KeccakMILPModel.chi_output_variable
    ),
    "KeccakMILPModel.chi_output_values": (
        KeccakMILPModel.chi_output_values
    ),
}


for name, obj in objects_to_inspect.items():
    print(
        f"{name}{inspect.signature(obj)}"
    )

chi(state: 'NDArray[np.integer]') -> 'NDArray[np.int64]'
KeccakMILPModel.add_chi_layer(self, round_index: 'int') -> 'None'
KeccakMILPModel.chi_output_variable(self, round_index: 'int', x: 'int', y: 'int', k: 'int') -> 'pulp.LpVariable'
KeccakMILPModel.chi_output_values(self, round_index: 'int', tolerance: 'float' = 0.5) -> 'list[list[list[int]]]'


## Formulación MILP de Chi

Sea $B[x,y,k]$ el estado obtenido después de aplicar las capas `rho`
y `pi`.

La capa `chi` se define mediante:

$$
C[x,y,k]
=
B[x,y,k]
\oplus
\left(
\neg B[(x+1)\bmod 5,y,k]
\land
B[(x+2)\bmod 5,y,k]
\right).
$$

Para simplificar la formulación, se definen:

$$
a = B[x,y,k],
$$

$$
b = B[(x+1)\bmod 5,y,k],
$$

$$
c = B[(x+2)\bmod 5,y,k].
$$

También se introduce una variable binaria auxiliar:

$$
t = \neg b \land c.
$$

Como $b$ es binaria:

$$
\neg b = 1-b.
$$

La conjunción se linealiza mediante:

$$
t \leq 1-b,
$$

$$
t \leq c,
$$

$$
t \geq c-b.
$$

Estas tres desigualdades obligan a que:

$$
t=(1-b)c.
$$

Finalmente, se introduce una variable binaria de paridad $q$ para
representar:

$$
d=a\oplus t.
$$

La ecuación MILP correspondiente es:

$$
a+t=d+2q.
$$

Por cada bit del estado se crean:

- una variable binaria $t$ para la conjunción;
- una variable binaria $d$ para la salida;
- una variable binaria $q$ para la paridad;
- tres restricciones para la conjunción;
- una restricción para el XOR.

In [5]:
# ============================================================
# TABLA DE VERDAD LOCAL DE CHI
# ============================================================

truth_table = []


for a in (0, 1):
    for b in (0, 1):
        for c in (0, 1):
            t = (1 - b) & c
            d = a ^ t

            truth_table.append(
                {
                    "a": a,
                    "b": b,
                    "c": c,
                    "not_b_and_c": t,
                    "salida": d,
                }
            )


print("a  b  c  (NOT b AND c)  salida")
print("-" * 38)

for row in truth_table:
    print(
        f"{row['a']}  "
        f"{row['b']}  "
        f"{row['c']}         "
        f"{row['not_b_and_c']}          "
        f"{row['salida']}"
    )

a  b  c  (NOT b AND c)  salida
--------------------------------------
0  0  0         0          0
0  0  1         1          1
0  1  0         0          0
0  1  1         0          0
1  0  0         0          1
1  0  1         1          0
1  1  0         0          1
1  1  1         0          1


In [6]:
# ============================================================
# PRUEBA MÍNIMA DE LA IMPLEMENTACIÓN DE REFERENCIA
# ============================================================

import numpy as np


z = 8

zero_state = np.zeros(
    (5, 5, z),
    dtype=np.int64,
)

chi_zero_state = chi(
    zero_state.copy()
)


assert isinstance(
    chi_zero_state,
    np.ndarray,
)

assert chi_zero_state.shape == (
    5,
    5,
    z,
)

assert np.all(
    np.isin(chi_zero_state, [0, 1])
)

assert np.array_equal(
    chi_zero_state,
    zero_state,
)


print("Prueba mínima completada correctamente.")
print("Forma de entrada:", zero_state.shape)
print("Forma de salida:", chi_zero_state.shape)
print("Peso de entrada:", int(zero_state.sum()))
print("Peso de salida:", int(chi_zero_state.sum()))

Prueba mínima completada correctamente.
Forma de entrada: (5, 5, 8)
Forma de salida: (5, 5, 8)
Peso de entrada: 0
Peso de salida: 0


## Tamaño esperado de la formulación

El estado contiene:

$$
25z
$$

bits.

Por cada bit, `chi` crea tres variables binarias:

$$
t[x,y,k],
$$

$$
C[x,y,k],
$$

$$
q[x,y,k].
$$

Por tanto, la cantidad de variables añadidas es:

$$
N_{\mathrm{variables}}^{\chi}
=
3(25z)
=
75z.
$$

También se agregan cuatro restricciones por bit:

$$
N_{\mathrm{restricciones}}^{\chi}
=
4(25z)
=
100z.
$$

Para $z=8$, los valores esperados son:

$$
N_{\mathrm{variables}}^{\chi}=600,
$$

y:

$$
N_{\mathrm{restricciones}}^{\chi}=800.
$$

In [7]:
# ============================================================
# MODELO BASE: THETA SEGUIDO DE RHO-PI
# ============================================================

from keccak_milp.config import ExperimentConfig
from keccak_milp.model import KeccakMILPModel


z = 8
round_index = 0

config = ExperimentConfig(
    z=z,
    rounds=1,
    solver="cbc",
    verbose=False,
)

model_until_rho_pi = KeccakMILPModel(
    config
)

model_until_rho_pi.add_theta_layer(
    round_index
)

model_until_rho_pi.add_rho_pi_layers(
    round_index
)


declared_before_chi = (
    model_until_rho_pi.declared_variable_count()
)

attached_before_chi = (
    model_until_rho_pi.attached_variable_count()
)

constraints_before_chi = (
    model_until_rho_pi.constraint_count()
)


print("Modelo construido hasta Rho-Pi")
print("-" * 45)
print(
    "Variables declaradas:",
    declared_before_chi,
)
print(
    "Variables conectadas:",
    attached_before_chi,
)
print(
    "Restricciones:",
    constraints_before_chi,
)

Modelo construido hasta Rho-Pi
---------------------------------------------
Variables declaradas: 1160
Variables conectadas: 960
Restricciones: 480


In [8]:
# ============================================================
# INCORPORACIÓN DE LA CAPA CHI
# ============================================================

model_until_rho_pi.add_chi_layer(
    round_index
)


declared_after_chi = (
    model_until_rho_pi.declared_variable_count()
)

attached_after_chi = (
    model_until_rho_pi.attached_variable_count()
)

constraints_after_chi = (
    model_until_rho_pi.constraint_count()
)


print("Modelo construido hasta Chi")
print("-" * 45)
print(
    "Variables declaradas:",
    declared_after_chi,
)
print(
    "Variables conectadas:",
    attached_after_chi,
)
print(
    "Restricciones:",
    constraints_after_chi,
)

Modelo construido hasta Chi
---------------------------------------------
Variables declaradas: 1760
Variables conectadas: 1560
Restricciones: 1280


In [9]:
# ============================================================
# VALIDACIÓN DEL TAMAÑO DE CHI
# ============================================================

added_declared_variables = (
    declared_after_chi
    - declared_before_chi
)

added_attached_variables = (
    attached_after_chi
    - attached_before_chi
)

added_constraints = (
    constraints_after_chi
    - constraints_before_chi
)


expected_variables = 3 * 25 * z
expected_constraints = 4 * 25 * z


print("Incremento producido por Chi")
print("-" * 45)
print(
    "Variables declaradas añadidas:",
    added_declared_variables,
)
print(
    "Variables conectadas añadidas:",
    added_attached_variables,
)
print(
    "Restricciones añadidas:",
    added_constraints,
)
print(
    "Variables esperadas:",
    expected_variables,
)
print(
    "Restricciones esperadas:",
    expected_constraints,
)


assert (
    added_declared_variables
    == expected_variables
)

assert (
    added_attached_variables
    == expected_variables
)

assert (
    added_constraints
    == expected_constraints
)


print(
    "\nLa formulación Chi tiene "
    "el tamaño esperado."
)

Incremento producido por Chi
---------------------------------------------
Variables declaradas añadidas: 600
Variables conectadas añadidas: 600
Restricciones añadidas: 800
Variables esperadas: 600
Restricciones esperadas: 800

La formulación Chi tiene el tamaño esperado.


In [10]:
# ============================================================
# VALIDACIÓN DE IDEMPOTENCIA
# ============================================================

variables_before_second_call = (
    model_until_rho_pi.declared_variable_count()
)

constraints_before_second_call = (
    model_until_rho_pi.constraint_count()
)


model_until_rho_pi.add_chi_layer(
    round_index
)


variables_after_second_call = (
    model_until_rho_pi.declared_variable_count()
)

constraints_after_second_call = (
    model_until_rho_pi.constraint_count()
)


assert (
    variables_after_second_call
    == variables_before_second_call
)

assert (
    constraints_after_second_call
    == constraints_before_second_call
)


print("Idempotencia verificada correctamente.")
print(
    "Variables antes y después:",
    variables_after_second_call,
)
print(
    "Restricciones antes y después:",
    constraints_after_second_call,
)

Idempotencia verificada correctamente.
Variables antes y después: 1760
Restricciones antes y después: 1280


In [11]:
# ============================================================
# INSPECCIÓN DE UNA VARIABLE DE SALIDA
# ============================================================

sample_chi_variable = (
    model_until_rho_pi.chi_output_variable(
        round_index=0,
        x=0,
        y=0,
        k=0,
    )
)

print(
    "Variable seleccionada:",
    sample_chi_variable.name,
)

print(
    "Categoría:",
    sample_chi_variable.cat,
)

print(
    "Límite inferior:",
    sample_chi_variable.lowBound,
)

print(
    "Límite superior:",
    sample_chi_variable.upBound,
)

Variable seleccionada: chi_output_r0_x0_y0_k0
Categoría: Integer
Límite inferior: 0
Límite superior: 1


## Validación funcional de la capa Chi

La validación estructural confirmó que la capa `chi` crea la cantidad
esperada de variables y restricciones. El siguiente paso consiste en
comprobar que la solución MILP reproduce exactamente la implementación
de referencia.

Se utilizará un estado binario controlado y se calcularán las salidas:

$$
T_{\mathrm{ref}}=\theta(A),
$$

$$
B_{\mathrm{ref}}=\rho\pi(T_{\mathrm{ref}}),
$$

$$
C_{\mathrm{ref}}=\chi(B_{\mathrm{ref}}).
$$

Luego se construirá el mismo flujo en el modelo MILP, se fijarán todos
los bits del estado inicial y se resolverá el problema con CBC.

La validación final exigirá:

$$
T_{\mathrm{MILP}}=T_{\mathrm{ref}},
$$

$$
B_{\mathrm{MILP}}=B_{\mathrm{ref}},
$$

y:

$$
C_{\mathrm{MILP}}=C_{\mathrm{ref}}.
$$

A diferencia de `rho` y `pi`, la capa `chi` no necesariamente conserva
el peso de Hamming, porque modifica los valores de los bits mediante una
operación no lineal.

In [12]:
# ============================================================
# FUNCIONES AUXILIARES PARA LA VALIDACIÓN
# ============================================================

from pulp import LpStatus


def hamming_weight(
    state: np.ndarray,
) -> int:
    """Calcula el peso de Hamming de un estado binario."""
    return int(
        np.asarray(state, dtype=np.int64).sum()
    )


def differing_positions(
    first_state: np.ndarray,
    second_state: np.ndarray,
) -> list[tuple[int, int, int, int, int]]:
    """
    Devuelve las posiciones donde dos estados son diferentes.

    Cada elemento contiene:

        (x, y, k, valor_primero, valor_segundo)
    """
    first_array = np.asarray(
        first_state,
        dtype=np.int64,
    )

    second_array = np.asarray(
        second_state,
        dtype=np.int64,
    )

    if first_array.shape != second_array.shape:
        raise ValueError(
            "Los estados deben tener la misma forma."
        )

    differences = []

    for x in range(first_array.shape[0]):
        for y in range(first_array.shape[1]):
            for k in range(first_array.shape[2]):
                first_value = int(
                    first_array[x, y, k]
                )

                second_value = int(
                    second_array[x, y, k]
                )

                if first_value != second_value:
                    differences.append(
                        (
                            x,
                            y,
                            k,
                            first_value,
                            second_value,
                        )
                    )

    return differences


def normalize_solution_state(
    state,
) -> np.ndarray:
    """Convierte una salida del modelo en un arreglo binario."""
    array = np.asarray(
        state,
        dtype=float,
    )

    if np.isnan(array).any():
        raise RuntimeError(
            "La solución contiene valores no definidos."
        )

    return np.rint(array).astype(
        np.int64
    )


print("Funciones auxiliares definidas correctamente.")

Funciones auxiliares definidas correctamente.


In [13]:
# ============================================================
# CONSTRUCCIÓN DE UNA ENTRADA CONTROLADA
# ============================================================

z = 8
round_index = 0

input_state = np.zeros(
    (5, 5, z),
    dtype=np.int64,
)

active_input_bits = [
    (0, 0, 0),
    (1, 0, 1),
    (0, 1, 3),
    (2, 3, 4),
    (3, 2, 5),
    (4, 4, 7),
]

for x, y, k in active_input_bits:
    input_state[x, y, k] = 1


assert input_state.shape == (5, 5, z)
assert np.all(np.isin(input_state, [0, 1]))
assert hamming_weight(input_state) > 0


print("Bits activos de la entrada:")

for position in active_input_bits:
    print(" -", position)

print(
    "\nPeso de Hamming de la entrada:",
    hamming_weight(input_state),
)

Bits activos de la entrada:
 - (0, 0, 0)
 - (1, 0, 1)
 - (0, 1, 3)
 - (2, 3, 4)
 - (3, 2, 5)
 - (4, 4, 7)

Peso de Hamming de la entrada: 6


In [14]:
# ============================================================
# CÁLCULO DE LAS SALIDAS DE REFERENCIA
# ============================================================

theta_reference = layers.theta(
    input_state.copy()
)

rho_pi_reference = layers.rho_pi(
    theta_reference.copy()
)

chi_reference = layers.chi(
    rho_pi_reference.copy()
)


assert theta_reference.shape == (5, 5, z)
assert rho_pi_reference.shape == (5, 5, z)
assert chi_reference.shape == (5, 5, z)


print("Pesos de Hamming de referencia")
print("-" * 45)
print(
    "Entrada:",
    hamming_weight(input_state),
)
print(
    "Después de Theta:",
    hamming_weight(theta_reference),
)
print(
    "Después de Rho-Pi:",
    hamming_weight(rho_pi_reference),
)
print(
    "Después de Chi:",
    hamming_weight(chi_reference),
)


assert (
    hamming_weight(theta_reference)
    ==
    hamming_weight(rho_pi_reference)
)

print(
    "\nRho-Pi conserva el peso de Hamming."
)
print(
    "Chi puede modificarlo por su carácter no lineal."
)

Pesos de Hamming de referencia
---------------------------------------------
Entrada: 6
Después de Theta: 66
Después de Rho-Pi: 66
Después de Chi: 89

Rho-Pi conserva el peso de Hamming.
Chi puede modificarlo por su carácter no lineal.


In [15]:
# ============================================================
# CONSTRUCCIÓN DEL MODELO THETA → RHO-PI → CHI
# ============================================================

validation_config = ExperimentConfig(
    z=z,
    rounds=1,
    solver="cbc",
    verbose=False,
)

validation_model = KeccakMILPModel(
    validation_config
)

# Registra la restricción de entrada no nula
# y la función objetivo del modelo.
validation_model.build_skeleton()

validation_model.add_theta_layer(
    round_index
)

validation_model.add_rho_pi_layers(
    round_index
)

validation_model.add_chi_layer(
    round_index
)


# ------------------------------------------------------------
# Fijar todos los bits de la entrada
# ------------------------------------------------------------

fixed_input_bits = 0

for x in range(5):
    for y in range(5):
        for k in range(z):
            input_variable = (
                validation_model.state_variable(
                    round_index=round_index,
                    x=x,
                    y=y,
                    k=k,
                )
            )

            input_value = int(
                input_state[x, y, k]
            )

            validation_model.problem += (
                input_variable == input_value,
                (
                    f"fix_validation_input"
                    f"_r{round_index}"
                    f"_x{x}_y{y}_k{k}"
                ),
            )

            fixed_input_bits += 1


assert fixed_input_bits == 25 * z


print("Modelo de validación construido.")
print("-" * 45)
print(
    "Variables declaradas:",
    validation_model.declared_variable_count(),
)
print(
    "Variables conectadas:",
    validation_model.attached_variable_count(),
)
print(
    "Restricciones:",
    validation_model.constraint_count(),
)
print(
    "Bits de entrada fijados:",
    fixed_input_bits,
)

Modelo de validación construido.
---------------------------------------------
Variables declaradas: 1760
Variables conectadas: 1560
Restricciones: 1481
Bits de entrada fijados: 200


In [16]:
# ============================================================
# RESOLUCIÓN DEL MODELO CON CBC
# ============================================================

solve_result = validation_model.solve()

status_code = validation_model.problem.status

status_name = LpStatus.get(
    status_code,
    str(status_code),
)


print(
    "Resultado devuelto por solve():",
    solve_result,
)

print(
    "Código de estado:",
    status_code,
)

print(
    "Estado del solver:",
    status_name,
)


assert status_name == "Optimal", (
    "Se esperaba una solución óptima, "
    f"pero CBC devolvió: {status_name}."
)

print(
    "\nEl modelo fue resuelto correctamente."
)

Resultado devuelto por solve(): Optimal
Código de estado: 1
Estado del solver: Optimal

El modelo fue resuelto correctamente.


In [17]:
# ============================================================
# RECUPERACIÓN DE LAS SALIDAS DEL MODELO
# ============================================================

theta_milp = normalize_solution_state(
    validation_model.theta_output_values(
        round_index
    )
)

rho_pi_milp = normalize_solution_state(
    validation_model.rho_pi_output_values(
        round_index
    )
)

chi_milp = normalize_solution_state(
    validation_model.chi_output_values(
        round_index
    )
)


print("Pesos de Hamming obtenidos por el MILP")
print("-" * 45)
print(
    "Salida de Theta:",
    hamming_weight(theta_milp),
)
print(
    "Salida de Rho-Pi:",
    hamming_weight(rho_pi_milp),
)
print(
    "Salida de Chi:",
    hamming_weight(chi_milp),
)

Pesos de Hamming obtenidos por el MILP
---------------------------------------------
Salida de Theta: 66
Salida de Rho-Pi: 66
Salida de Chi: 89


In [18]:
# ============================================================
# COMPARACIÓN MILP FRENTE A LA REFERENCIA
# ============================================================

theta_differences = differing_positions(
    theta_milp,
    theta_reference,
)

rho_pi_differences = differing_positions(
    rho_pi_milp,
    rho_pi_reference,
)

chi_differences = differing_positions(
    chi_milp,
    chi_reference,
)


print("Comparación bit a bit")
print("-" * 45)
print(
    "Diferencias en Theta:",
    len(theta_differences),
)
print(
    "Diferencias en Rho-Pi:",
    len(rho_pi_differences),
)
print(
    "Diferencias en Chi:",
    len(chi_differences),
)


if theta_differences:
    print("\nPrimeras diferencias de Theta:")

    for difference in theta_differences[:10]:
        print(difference)


if rho_pi_differences:
    print("\nPrimeras diferencias de Rho-Pi:")

    for difference in rho_pi_differences[:10]:
        print(difference)


if chi_differences:
    print("\nPrimeras diferencias de Chi:")

    for difference in chi_differences[:10]:
        print(difference)


assert not theta_differences, (
    "La formulación MILP de Theta no coincide "
    "con la implementación de referencia."
)

assert not rho_pi_differences, (
    "La formulación MILP de Rho-Pi no coincide "
    "con la implementación de referencia."
)

assert not chi_differences, (
    "La formulación MILP de Chi no coincide "
    "con la implementación de referencia."
)


print(
    "\nValidación funcional completada correctamente."
)

Comparación bit a bit
---------------------------------------------
Diferencias en Theta: 0
Diferencias en Rho-Pi: 0
Diferencias en Chi: 0

Validación funcional completada correctamente.


## Inspección de las variables auxiliares

Además de comparar la salida final, es útil verificar directamente las
variables auxiliares utilizadas en la formulación.

Para una posición $(x,y,k)$, el modelo define:

$$
a=B[x,y,k],
$$

$$
b=B[(x+1)\bmod 5,y,k],
$$

$$
c=B[(x+2)\bmod 5,y,k],
$$

$$
t=\neg b \land c,
$$

y:

$$
d=a\oplus t.
$$

La variable auxiliar de paridad $q$ debe satisfacer:

$$
a+t=d+2q.
$$

Esta comprobación permite observar que el solver no solamente produce la
salida correcta, sino que asigna valores coherentes a las variables internas
de la linealización.

In [19]:
# ============================================================
# INSPECCIÓN DE LAS VARIABLES AUXILIARES DE CHI
# ============================================================

sample_x = 0
sample_y = 0
sample_k = 0

next_x = (sample_x + 1) % 5
second_next_x = (sample_x + 2) % 5

sample_index = (
    round_index,
    sample_x,
    sample_y,
    sample_k,
)


# Entradas locales de Chi
a_value = int(
    rho_pi_milp[
        sample_x,
        sample_y,
        sample_k,
    ]
)

b_value = int(
    rho_pi_milp[
        next_x,
        sample_y,
        sample_k,
    ]
)

c_value = int(
    rho_pi_milp[
        second_next_x,
        sample_y,
        sample_k,
    ]
)


# Variables auxiliares recuperadas del modelo
and_raw_value = (
    validation_model.chi_and[
        sample_index
    ].value()
)

parity_raw_value = (
    validation_model.chi_q[
        sample_index
    ].value()
)

output_raw_value = (
    validation_model.chi_output[
        sample_index
    ].value()
)


assert and_raw_value is not None
assert parity_raw_value is not None
assert output_raw_value is not None


and_value = int(round(and_raw_value))
parity_value = int(round(parity_raw_value))
output_value = int(round(output_raw_value))


expected_and = (1 - b_value) & c_value
expected_output = a_value ^ expected_and


print("Posición analizada:", (sample_x, sample_y, sample_k))
print("-" * 45)
print("a =", a_value)
print("b =", b_value)
print("c =", c_value)
print("t = (NOT b) AND c =", and_value)
print("d = a XOR t =", output_value)
print("q =", parity_value)

print("\nVerificación de la ecuación de paridad:")
print(
    f"a + t = {a_value + and_value}"
)
print(
    f"d + 2q = {output_value + 2 * parity_value}"
)


assert and_value == expected_and
assert output_value == expected_output

assert (
    a_value + and_value
    ==
    output_value + 2 * parity_value
)

print(
    "\nLas variables auxiliares satisfacen "
    "la formulación de Chi."
)

Posición analizada: (0, 0, 0)
---------------------------------------------
a = 1
b = 0
c = 0
t = (NOT b) AND c = 0
d = a XOR t = 1
q = 0

Verificación de la ecuación de paridad:
a + t = 1
d + 2q = 1

Las variables auxiliares satisfacen la formulación de Chi.


## Validación sobre estados pseudoaleatorios

La prueba controlada valida una ejecución concreta del modelo. Para ampliar
la evidencia experimental, se repetirán las comparaciones sobre varios
estados binarios pseudoaleatorios.

Se utilizará una semilla fija para que los resultados puedan reproducirse.

Para cada caso se verificará:

$$
T_{\mathrm{MILP}}=T_{\mathrm{ref}},
$$

$$
B_{\mathrm{MILP}}=B_{\mathrm{ref}},
$$

$$
C_{\mathrm{MILP}}=C_{\mathrm{ref}}.
$$

También se registrarán los pesos de Hamming en cada etapa. Se espera que
`rho_pi` conserve el peso de `theta`, mientras que `chi` puede modificarlo.

In [20]:
# ============================================================
# FUNCIÓN REUTILIZABLE: THETA → RHO-PI → CHI
# ============================================================

def solve_theta_rho_pi_chi_case(
    input_state: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, str]:
    """
    Resuelve mediante MILP las capas Theta, Rho-Pi y Chi para
    un estado de entrada completamente fijado.

    Parameters
    ----------
    input_state:
        Estado binario NumPy con forma (5, 5, z).

    Returns
    -------
    tuple
        Salida de Theta, salida de Rho-Pi, salida de Chi y
        estado textual del solver.
    """
    input_array = np.asarray(
        input_state,
        dtype=np.int64,
    )

    if input_array.ndim != 3:
        raise ValueError(
            "El estado debe tener tres dimensiones."
        )

    if input_array.shape[:2] != (5, 5):
        raise ValueError(
            "Las dos primeras dimensiones deben ser 5 × 5."
        )

    local_z = input_array.shape[2]

    if local_z not in {4, 8}:
        raise ValueError(
            "El tamaño de palabra debe ser 4 u 8."
        )

    if not np.all(
        np.isin(input_array, [0, 1])
    ):
        raise ValueError(
            "El estado debe contener únicamente valores binarios."
        )

    local_round = 0

    local_config = ExperimentConfig(
        z=local_z,
        rounds=1,
        solver="cbc",
        verbose=False,
    )

    local_model = KeccakMILPModel(
        local_config
    )

    local_model.add_theta_layer(
        local_round
    )

    local_model.add_rho_pi_layers(
        local_round
    )

    local_model.add_chi_layer(
        local_round
    )


    # --------------------------------------------------------
    # Fijar completamente el estado inicial
    # --------------------------------------------------------

    for x in range(5):
        for y in range(5):
            for k in range(local_z):
                input_variable = (
                    local_model.state_variable(
                        round_index=local_round,
                        x=x,
                        y=y,
                        k=k,
                    )
                )

                local_model.problem += (
                    input_variable
                    == int(input_array[x, y, k]),
                    (
                        f"fix_random_input"
                        f"_x{x}_y{y}_k{k}"
                    ),
                )


    # Registra el objetivo requerido por solve().
    #
    # No se usa build_skeleton() porque el estado ya está
    # completamente fijado y también queremos admitir el caso
    # de entrada nula.
    local_model.set_smoke_test_objective()

    status = local_model.solve()

    if status != "Optimal":
        raise RuntimeError(
            "CBC no encontró una solución óptima. "
            f"Estado obtenido: {status}."
        )


    theta_output = normalize_solution_state(
        local_model.theta_output_values(
            local_round
        )
    )

    rho_pi_output = normalize_solution_state(
        local_model.rho_pi_output_values(
            local_round
        )
    )

    chi_output = normalize_solution_state(
        local_model.chi_output_values(
            local_round
        )
    )


    return (
        theta_output,
        rho_pi_output,
        chi_output,
        status,
    )


print(
    "Función reutilizable definida correctamente."
)

Función reutilizable definida correctamente.


In [21]:
# ============================================================
# PRUEBA DE LA FUNCIÓN CON EL CASO CONTROLADO
# ============================================================

(
    theta_function_output,
    rho_pi_function_output,
    chi_function_output,
    function_status,
) = solve_theta_rho_pi_chi_case(
    input_state
)


assert np.array_equal(
    theta_function_output,
    theta_reference,
)

assert np.array_equal(
    rho_pi_function_output,
    rho_pi_reference,
)

assert np.array_equal(
    chi_function_output,
    chi_reference,
)


print("Estado del solver:", function_status)
print(
    "Peso de Theta:",
    hamming_weight(theta_function_output),
)
print(
    "Peso de Rho-Pi:",
    hamming_weight(rho_pi_function_output),
)
print(
    "Peso de Chi:",
    hamming_weight(chi_function_output),
)

print(
    "\nLa función reutilizable reproduce "
    "correctamente el caso controlado."
)

Estado del solver: Optimal
Peso de Theta: 66
Peso de Rho-Pi: 66
Peso de Chi: 89

La función reutilizable reproduce correctamente el caso controlado.


In [22]:
# ============================================================
# VALIDACIÓN SOBRE ESTADOS PSEUDOALEATORIOS
# ============================================================

random_seed = 2026
number_of_cases = 5
z = 8

rng = np.random.default_rng(
    random_seed
)

random_validation_results = []


for case_index in range(number_of_cases):
    random_input = rng.integers(
        low=0,
        high=2,
        size=(5, 5, z),
        dtype=np.int64,
    )


    # --------------------------------------------------------
    # Implementación de referencia
    # --------------------------------------------------------

    theta_expected = layers.theta(
        random_input.copy()
    )

    rho_pi_expected = layers.rho_pi(
        theta_expected.copy()
    )

    chi_expected = layers.chi(
        rho_pi_expected.copy()
    )


    # --------------------------------------------------------
    # Implementación MILP
    # --------------------------------------------------------

    (
        theta_obtained,
        rho_pi_obtained,
        chi_obtained,
        solver_status,
    ) = solve_theta_rho_pi_chi_case(
        random_input
    )


    # --------------------------------------------------------
    # Comparaciones
    # --------------------------------------------------------

    theta_correct = np.array_equal(
        theta_obtained,
        theta_expected,
    )

    rho_pi_correct = np.array_equal(
        rho_pi_obtained,
        rho_pi_expected,
    )

    chi_correct = np.array_equal(
        chi_obtained,
        chi_expected,
    )

    rho_pi_weight_preserved = (
        hamming_weight(theta_obtained)
        ==
        hamming_weight(rho_pi_obtained)
    )


    random_validation_results.append(
        {
            "caso": case_index + 1,
            "solver": solver_status,
            "peso_entrada": hamming_weight(
                random_input
            ),
            "peso_theta": hamming_weight(
                theta_obtained
            ),
            "peso_rho_pi": hamming_weight(
                rho_pi_obtained
            ),
            "peso_chi": hamming_weight(
                chi_obtained
            ),
            "theta_correcto": theta_correct,
            "rho_pi_correcto": rho_pi_correct,
            "chi_correcto": chi_correct,
            "peso_rho_pi_conservado": (
                rho_pi_weight_preserved
            ),
        }
    )


    assert solver_status == "Optimal"
    assert theta_correct
    assert rho_pi_correct
    assert chi_correct
    assert rho_pi_weight_preserved


print(
    f"Se validaron correctamente "
    f"{number_of_cases} estados pseudoaleatorios."
)

Se validaron correctamente 5 estados pseudoaleatorios.


In [23]:
# ============================================================
# RESUMEN DE LOS CASOS PSEUDOALEATORIOS
# ============================================================

header = (
    "Caso | Solver  | Entrada | Theta | Rho-Pi | Chi | "
    "Theta OK | Rho-Pi OK | Chi OK"
)

print(header)
print("-" * len(header))


for result in random_validation_results:
    print(
        f"{result['caso']:>4} | "
        f"{result['solver']:<7} | "
        f"{result['peso_entrada']:>7} | "
        f"{result['peso_theta']:>5} | "
        f"{result['peso_rho_pi']:>6} | "
        f"{result['peso_chi']:>3} | "
        f"{str(result['theta_correcto']):>8} | "
        f"{str(result['rho_pi_correcto']):>9} | "
        f"{str(result['chi_correcto']):>6}"
    )

Caso | Solver  | Entrada | Theta | Rho-Pi | Chi | Theta OK | Rho-Pi OK | Chi OK
-------------------------------------------------------------------------------
   1 | Optimal |      97 |    93 |     93 |  89 |     True |      True |   True
   2 | Optimal |      95 |   105 |    105 |  95 |     True |      True |   True
   3 | Optimal |      95 |    97 |     97 | 101 |     True |      True |   True
   4 | Optimal |     100 |   104 |    104 | 100 |     True |      True |   True
   5 | Optimal |      95 |   107 |    107 | 105 |     True |      True |   True


In [24]:
# ============================================================
# COMPROBACIÓN GLOBAL
# ============================================================

all_solver_optimal = all(
    result["solver"] == "Optimal"
    for result in random_validation_results
)

all_theta_correct = all(
    result["theta_correcto"]
    for result in random_validation_results
)

all_rho_pi_correct = all(
    result["rho_pi_correcto"]
    for result in random_validation_results
)

all_chi_correct = all(
    result["chi_correcto"]
    for result in random_validation_results
)

all_rho_pi_weights_preserved = all(
    result["peso_rho_pi_conservado"]
    for result in random_validation_results
)


assert all_solver_optimal
assert all_theta_correct
assert all_rho_pi_correct
assert all_chi_correct
assert all_rho_pi_weights_preserved


print("Todas las validaciones fueron superadas.")
print("-" * 45)
print(
    "Casos evaluados:",
    len(random_validation_results),
)
print(
    "Errores de Theta:",
    sum(
        not result["theta_correcto"]
        for result in random_validation_results
    ),
)
print(
    "Errores de Rho-Pi:",
    sum(
        not result["rho_pi_correcto"]
        for result in random_validation_results
    ),
)
print(
    "Errores de Chi:",
    sum(
        not result["chi_correcto"]
        for result in random_validation_results
    ),
)
print(
    "Violaciones de conservación en Rho-Pi:",
    sum(
        not result["peso_rho_pi_conservado"]
        for result in random_validation_results
    ),
)

Todas las validaciones fueron superadas.
---------------------------------------------
Casos evaluados: 5
Errores de Theta: 0
Errores de Rho-Pi: 0
Errores de Chi: 0
Violaciones de conservación en Rho-Pi: 0


## Conclusiones

La capa no lineal `chi` fue incorporada correctamente al modelo MILP de
Keccak reducido.

Los principales resultados son:

1. La implementación de referencia reproduce la expresión:

   $$
   C[x,y,k]
   =
   B[x,y,k]
   \oplus
   \left(
   \neg B[(x+1)\bmod 5,y,k]
   \land
   B[(x+2)\bmod 5,y,k]
   \right).
   $$

2. La conjunción no lineal se representa mediante una variable auxiliar:

   $$
   t=(1-b)c,
   $$

   y tres desigualdades lineales.

3. El XOR entre $a$ y $t$ se representa mediante:

   $$
   a+t=d+2q.
   $$

4. Por cada bit del estado se crean tres variables binarias y cuatro
   restricciones.

5. Para $z=8$, la capa agrega:

   $$
   600
   $$

   variables binarias y:

   $$
   800
   $$

   restricciones.

6. El método `add_chi_layer` es idempotente y requiere que las capas
   `rho` y `pi` hayan sido agregadas previamente.

7. Las variables auxiliares obtenidas por CBC satisfacen tanto la
   linealización de la conjunción como la ecuación de paridad.

8. Las salidas MILP de `theta`, `rho_pi` y `chi` coinciden bit a bit con
   las implementaciones de referencia.

9. La equivalencia fue comprobada para una entrada controlada y para varios
   estados pseudoaleatorios reproducibles.

10. A diferencia de `rho` y `pi`, `chi` puede modificar el peso de Hamming,
    debido a su carácter no lineal.

Con esta etapa, el modelo representa correctamente el flujo:

$$
A
\longrightarrow
\theta
\longrightarrow
\rho
\longrightarrow
\pi
\longrightarrow
\chi.
$$

La siguiente etapa será incorporar `iota`, conectar la salida de una ronda
con el siguiente estado de frontera y completar una ronda reducida de
Keccak.